## Street Slope Calculation via DTM
In this section, we use the 2-meter resolution Digital Terrain Model (DTM) from ICGC to extract the elevation at the start and end of every street segment. This allows us to calculate the percentage slope, which is a critical factor for vehicle acoustic emissions.

In [20]:
import geopandas as gpd
import pandas as pd
import rasterio
import numpy as np
from tqdm.notebook import tqdm

# Load the street network geometry layer
noise_streets = gpd.read_file("../layers/BCN_noise_streets.gpkg")

print("Libraries and spatial data loaded. Ready for the DTM!")

Libraries and spatial data loaded. Ready for the DTM!


In [21]:
# Define the path to your TIF file. 
# Remember: the .tfw file MUST be in the exact same folder and have the exact same name!
dtm_path = "../layers/Digital_terrain_model/elevacions-terreny-lidar-Catalunya-2m-2008-2011tif1777940893149.tif"

In [22]:
print("--- CONTROLLO ALLINEAMENTO SPAZIALE ---")

# Controllo coordinate Strade
print("1. CRS Strade:", noise_streets.crs)
print("2. Confini Strade (Min X, Min Y, Max X, Max Y):")
print(noise_streets.total_bounds)

print("-" * 30)

# Controllo coordinate DTM
with rasterio.open(dtm_path) as src:
    print("3. CRS DTM:", src.crs)
    print("4. Confini DTM (Min X, Min Y, Max X, Max Y):")
    print(src.bounds)

--- CONTROLLO ALLINEAMENTO SPAZIALE ---
1. CRS Strade: EPSG:25831
2. Confini Strade (Min X, Min Y, Max X, Max Y):
[ 421364.69074511 4574934.61324795  435002.22300797 4591063.62179738]
------------------------------
3. CRS DTM: EPSG:25831
4. Confini DTM (Min X, Min Y, Max X, Max Y):
BoundingBox(left=420696.0, bottom=4574240.0, right=435564.0, top=4591138.0)


### Extracting Elevations (Z) from the Raster
We open the DTM and "sample" the pixels that correspond to the coordinates of the first and last node of every street (LineString). We use a `try-except` block to prevent the code from crashing if a street goes outside the boundaries of our raster file.

In [23]:
z_start_list = []
z_end_list = []

print("Sampling elevations (Handling MultiLineStrings)...")

with rasterio.open(dtm_path) as src:
    for geom in tqdm(noise_streets.geometry, desc="Processing Streets"):
        
        try:
            # 1. Check if it's a standard line
            if geom.geom_type == 'LineString':
                start_coord = geom.coords[0]
                end_coord = geom.coords[-1]
                
            # 2. Check if it's a MultiLineString (OSM standard)
            elif geom.geom_type == 'MultiLineString':
                start_coord = geom.geoms[0].coords[0]    # First point of the first line
                end_coord = geom.geoms[-1].coords[-1]    # Last point of the last line
                
            # 3. If it's a polygon or point, skip it
            else:
                z_start_list.append(np.nan)
                z_end_list.append(np.nan)
                continue
            
            # Sample the DTM exactly at those X/Y coordinates
            z_start = next(src.sample([(start_coord[0], start_coord[1])]))[0]
            z_end = next(src.sample([(end_coord[0], end_coord[1])]))[0]
            
            # Note: The DTM might return a negative number (e.g., -9999) for nodata areas. 
            # We filter those out just in case.
            if z_start < -100 or z_end < -100:
                z_start, z_end = np.nan, np.nan
                
        except Exception as e:
            # If the point is outside the raster or throws an error, return NaN
            z_start, z_end = np.nan, np.nan
            
        z_start_list.append(z_start)
        z_end_list.append(z_end)

# Add the extracted values back to the GeoDataFrame
noise_streets['elev_start_m'] = z_start_list
noise_streets['elev_end_m'] = z_end_list

print("Elevations extracted successfully!")

Sampling elevations (Handling MultiLineStrings)...


Processing Streets:   0%|          | 0/15115 [00:00<?, ?it/s]

Elevations extracted successfully!


### Calculating Percentage Slope
We calculate the exact length of each segment in meters and apply the absolute slope formula: `|Z_end - Z_start| / Length * 100`

In [24]:
# 1. Calculate the street length
noise_streets['length_m'] = noise_streets.geometry.length

# 2. Apply the slope formula (absolute value)
noise_streets['slope_pct'] = (abs(noise_streets['elev_end_m'] - noise_streets['elev_start_m']) / noise_streets['length_m']) * 100

# 3. Data cleaning: replace NaNs (streets off-map or errors) with a 0 slope
noise_streets['slope_pct'] = noise_streets['slope_pct'].fillna(0)

# 4. Visual check of the results
print("Slope calculation complete!")
display(noise_streets[['length_m', 'elev_start_m', 'elev_end_m', 'slope_pct']].head())

Slope calculation complete!


,length_m,elev_start_m,elev_end_m,slope_pct
0,48.661095,74.180000,74.620003,0.904218
1,87.202693,3.950000,3.520000,0.493104
2,79.934562,119.220001,120.610001,1.738922
3,77.367711,25.750000,30.200001,5.751754
4,107.679649,4.990000,8.840000,3.575421


## Final Feature Dataset
Merge the extracted slope percentage with our target noise variables.

In [25]:
# Create a final clean dataset with our core target variables and the new feature
dataset = pd.DataFrame({
    "street_id": noise_streets['TRAM'],
    "noise_day": noise_streets['TOTAL_D'],
    "noise_evening": noise_streets['TOTAL_E'],
    "noise_night": noise_streets['TOTAL_N'],
    "slope_pct": noise_streets['slope_pct']
}).fillna(0)

display(dataset.head(10))

,street_id,noise_day,noise_evening,noise_night,slope_pct
0,T04719W,70 - 75 dB(A),65 - 70 dB(A),60 - 65 dB(A),0.904218
1,T19941Z,45 - 50 dB(A),45 - 50 dB(A),< 40 dB(A),0.493104
2,T18111R,55 - 60 dB(A),55 - 60 dB(A),50 - 55 dB(A),1.738922
3,T03222Y,60 - 65 dB(A),55 - 60 dB(A),60 - 65 dB(A),5.751754
4,T17625I,55 - 60 dB(A),55 - 60 dB(A),50 - 55 dB(A),3.575421
5,T05360P,55 - 60 dB(A),55 - 60 dB(A),50 - 55 dB(A),1.318660
6,T08863T,50 - 55 dB(A),50 - 55 dB(A),45 - 50 dB(A),3.131140
7,T00236S,45 - 50 dB(A),45 - 50 dB(A),40 - 45 dB(A),0.174952
8,T13009A,55 - 60 dB(A),55 - 60 dB(A),50 - 55 dB(A),4.989835
9,T11921P,60 - 65 dB(A),55 - 60 dB(A),50 - 55 dB(A),6.317352
